# Isolation Forest Anomaly Detector

This notebook introduces a machine-learning approach to detecting anomalies in the CATS telemetry dataset using Isolation Forest.

Unlike the rule-based and rolling statistical detectors, which use predefined thresholds, Isolation Forest learns patterns from the input data and assigns anomaly scores based on how easily observations can be isolated.

The model will use the 17 telemetry signals as input features. The ground-truth label `y`, the event `category`, and event metadata will be excluded from the model features. These fields will be reserved for evaluation.

The first 1,000,000 observations form the nominal period, while the remaining 4,000,000 observations form the evaluation period. We will use the nominal period to establish the model-fitting strategy and evaluate predictions on the later period without using its ground-truth labels to generate predictions.

The model will be evaluated using the same observation-level metrics as the earlier detectors, including precision, recall, and false-alarm rate. Its performance will also be assessed at the event level.

The objective is to determine whether Isolation Forest improves anomaly detection compared with the rule-based and rolling statistical baselines.

## Step 2 — Verify Dataset Files and Required Packages

This step verifies the locations of the CATS telemetry dataset and its event metadata, and checks whether the required machine-learning and data-processing packages are installed.

We will inspect the Parquet file's metadata to confirm its row count and columns without loading the complete dataset.

This verification helps establish that the files and dependencies are available before we prepare the model inputs. No model fitting or prediction will occur in this step.

In [3]:

from pathlib import Path
from importlib.metadata import version

import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "metadata.csv"

# print("Project root:", PROJECT_ROOT)
print("Telemetry file exists:", DATA_PATH.exists())
print("Metadata file exists:", METADATA_PATH.exists())

if DATA_PATH.exists():
    parquet_file = pq.ParquetFile(DATA_PATH)

    print("Dataset rows:", parquet_file.metadata.num_rows)
    print("Dataset columns:", parquet_file.schema_arrow.names)
    print("Telemetry file size (GB):",
          round(DATA_PATH.stat().st_size / (1024**3), 2))

if METADATA_PATH.exists():
    print("Metadata file size (MB):",
          round(METADATA_PATH.stat().st_size / (1024**2), 2))

for package in ["numpy", "pandas", "scikit-learn", "pyarrow"]:
    try:
        print(f"{package} version:", version(package))
    except Exception:
        print(f"{package}: NOT INSTALLED")


Telemetry file exists: True
Metadata file exists: True
Dataset rows: 5000000
Dataset columns: ['aimp', 'amud', 'arnd', 'asin1', 'asin2', 'adbr', 'adfl', 'bed1', 'bed2', 'bfo1', 'bfo2', 'bso1', 'bso2', 'bso3', 'ced1', 'cfo1', 'cso1', 'y', 'category', '__index_level_0__']
Telemetry file size (GB): 0.52
Metadata file size (MB): 0.01
numpy version: 2.4.6
pandas version: 3.0.6
scikit-learn version: 1.9.1
pyarrow version: 25.0.1


### Conclusion

The CATS telemetry file and event metadata are present at the expected project paths. The Parquet metadata confirms that the dataset contains 5,000,000 observations, and all required packages are installed.

The telemetry file includes 17 signals, the ground-truth label `y`, the event category, and the stored index column. The timestamp index must be preserved during data loading, while only the 17 telemetry signals will be used as model features.

The local verification is complete. The next step is to prepare the Kaggle execution environment, where we will have a better opportunity to process the dataset without exhausting the laptop's available memory.

## Step 3 — Load the Dataset and Prepare Model Inputs

We will load the CATS telemetry dataset from the Kaggle input directory and separate the 17 telemetry signals from the ground-truth labels and event categories.

The first 1,000,000 observations will be used as the nominal training period, and the remaining 4,000,000 observations will be reserved for evaluation.

The Isolation Forest model will use only the telemetry signals. The ground-truth label `y` and event category will be retained for evaluation but excluded from model training.

We will preserve the timestamp index to support event-level evaluation and detection-delay analysis.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path(
    "/kaggle/input/datasets/patrickfleith/"
    "controlled-anomalies-time-series-dataset"
)

for path in INPUT_ROOT.rglob("*"):
    if path.is_file():
        print(path)

## Step 3B — Load the CATS Dataset

The mounted Kaggle dataset provides the telemetry data as `data.csv` and the event metadata as `metadata.csv`.

We will load the telemetry data, preserve its time index, verify the required columns, and separate the first 1,000,000 observations from the remaining 4,000,000 observations. Only the 17 telemetry signals will be used as model features; labels and categories will be reserved for evaluation.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_ROOT = Path(
    "/kaggle/input/datasets/patrickfleith/"
    "controlled-anomalies-time-series-dataset"
)

DATA_PATH = INPUT_ROOT / "data.csv"
METADATA_PATH = INPUT_ROOT / "metadata.csv"

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1"
]

# Read the telemetry CSV.
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=True)
event_metadata = pd.read_csv(METADATA_PATH)

required_columns = signal_columns + ["y", "category"]
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if len(df) != 5_000_000:
    raise ValueError(f"Unexpected row count: {len(df):,}")

X_train = df.iloc[:1_000_000][signal_columns]
X_eval = df.iloc[1_000_000:][signal_columns]

y_eval = df.iloc[1_000_000:]["y"].astype("int8")
category_eval = df.iloc[1_000_000:]["category"]

print("Dataset shape:", df.shape)
print("Training shape:", X_train.shape)
print("Evaluation shape:", X_eval.shape)
print("Index type:", type(df.index).__name__)
print("First timestamp:", df.index.min())
print("Last timestamp:", df.index.max())
print("Event metadata shape:", event_metadata.shape)
print("Missing training values:", int(X_train.isna().sum().sum()))
print("Missing evaluation values:", int(X_eval.isna().sum().sum()))
print("Training label counts:")
print(df.iloc[:1_000_000]["y"].value_counts(dropna=False))

### Conclusion

The CATS dataset loaded successfully in Kaggle with 5,000,000 observations, 17 telemetry signals, a preserved datetime index, and 200 event records in the metadata.

The first 1,000,000 observations contain only normal labels, and neither the training nor evaluation telemetry contains missing values. The training and evaluation feature matrices have the expected dimensions.

The data is ready for the Isolation Forest fitting strategy. The model will be fitted using normal-period telemetry features, while the later four million observations and their ground-truth labels will remain reserved for evaluation.

## Step 4 — Configure Isolation Forest

Isolation Forest is an unsupervised anomaly-detection algorithm that identifies unusual observations by recursively partitioning the feature space. Anomalies tend to be isolated in fewer splits than observations that follow common patterns.

### 4.1 Mathematical foundation of Isolation Forest

**Average path length**

Let \(h_t(x)\) represent the number of splits required to isolate observation \(x\) in tree \(t\). For a forest containing \(T\) trees, the average path length is

$$
E[h(x)] = \frac{1}{T}\sum_{t=1}^{T}h_t(x)
$$

where:

- \(x\) is an observation represented by the 17 telemetry signals.
- \(h_t(x)\) is its path length in tree \(t\).
- \(T\) is the total number of trees.
- \(E[h(x)]\) is the average path length across the forest.

An observation that is isolated quickly generally has a shorter average path length and is more likely to be anomalous.

**Anomaly score**

Isolation Forest normalizes the average path length to calculate an anomaly score:

$$
s(x,n)=2^{-\frac{E[h(x)]}{c(n)}}
$$

where \(n\) is the number of observations used to build each tree, and \(c(n)\) is the expected path length used for normalization.

For \(n>2\), the normalization term is

$$
c(n)=2H(n-1)-\frac{2(n-1)}{n}
$$

Here, \(H(n-1)\) is the harmonic number

$$
H(n-1)=\sum_{k=1}^{n-1}\frac{1}{k}
$$

For large \(n\), the harmonic number can be approximated by

$$
H(n-1)\approx \ln(n-1)+\gamma
$$

where \(\ln\) is the natural logarithm and \(\gamma\approx0.57721\) is the Euler–Mascheroni constant.

The score generally ranges between 0 and 1:

- Scores closer to 1 indicate observations that are easier to isolate.
- Scores around 0.5 are near the theoretical reference point of the normalized score.
- Lower scores generally indicate observations that require longer paths to isolate.

These interpretations describe the scoring mechanism; the appropriate decision threshold must still be determined for our anomaly-detection task.

### 4.2 Training data and model parameters

The first 1,000,000 observations form our nominal training period. All of these observations have ground-truth label \(y=0\). We will use the 17 telemetry signals as model features and exclude the ground-truth labels and event categories from training.

We will configure two principal parameters:

- **`n_estimators`**: the number of isolation trees. More trees can stabilize anomaly scores, but increase computation.
- **`max_samples`**: the maximum number of observations used to build each tree. This controls the sample size per tree, even when the model is fitted using the full training matrix.

Our initial configuration is

$$
N=1{,}000{,}000,\qquad T=300,\qquad m=2{,}048
$$

where \(N\) is the total number of nominal training observations, \(T\) is the number of trees, and \(m\) is the maximum number of observations used by each tree.

The maximum number of training-row selections across all trees is therefore

$$
T\times m=300\times2{,}048=614{,}400
$$

This is a count of selections across trees, not necessarily 614,400 unique observations, because different trees can use overlapping subsets. The model receives the full training matrix, but each tree uses its own subsample.

We will begin with 300 trees and 2,048 observations per tree as an initial configuration, not as a proven optimum. We will assess the resulting model and, if warranted, compare it with configurations that use more observations per tree or additional trees.

### 4.3 Evaluation and leakage prevention

The later 4,000,000 observations will remain reserved for evaluation. Their labels and event categories will not be used to fit the model or tune its decision threshold.

We will evaluate observation-level precision, recall, and false-alarm rate, as well as event-level detection and detection delay. We will also record training and prediction time so that performance can be assessed alongside computational cost.

The objective is not simply to maximize the number of training observations. It is to determine which model configuration provides reliable anomaly detection on unseen observations without using evaluation labels to influence model selection.

In [ ]:
from sklearn.ensemble import IsolationForest

isolation_forest = IsolationForest(
    n_estimators=300,
    max_samples=2048,
    contamination="auto",
    max_features=1.0,
    bootstrap=False,
    n_jobs=1,
    random_state=42
)

print("Training observations:", len(X_train))
print("Features:", X_train.shape[1])
print("Trees:", isolation_forest.n_estimators)
print("Maximum samples per tree:", isolation_forest.max_samples)
print("Parallel CPU workers:", isolation_forest.n_jobs)